# Stack Overflow Data Collection

In [1]:
%pip install --upgrade pip
%pip install -q requests beautifulsoup4 lxml pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Import necessary dependencies

In [2]:
import csv, os, time, json
from datetime import datetime, timedelta, timezone
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Define Constants

In [3]:
CONFIG = json.load(open("config.json"))

START_DATE = "2023-01-01"
END_DATE = "2026-11-25"
OUTPUT_DIR = os.path.dirname(CONFIG["src_path"])
OUTPUT_CSV_PATH = CONFIG["src_path"]
API = "https://api.stackexchange.com/2.3"
API_KEY = "" # Replace API key with valid stack exchange API which allow 10,000 requests per day
TAGS = CONFIG["domain_tags"]

## Helper functions

In [4]:
def clean_question_body(html):
    if not html:
        return "", ""
    soup = BeautifulSoup(html, "lxml")
    pres = soup.find_all("pre")
    code = "\n\n".join(pre.get_text("\n", strip=True) for pre in pres)
    for pre in pres:
        pre.decompose()
    text = " ".join(soup.get_text(" ", strip=True).split())
    return text, code

def load_seen_question_ids(file_path):
    if not os.path.exists(file_path):
        return set()
    with open(file_path, newline="", encoding="utf-8") as file:
        return {row["question_id"] for row in csv.DictReader(file)}

def append_rows(path, rows):
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as file:
        w = csv.DictWriter(file, fieldnames=[
            "question_id",
            "title",
            "body_text",
            "code_text",
            "tags",
            "score",
            "view_count",
            "answer_count",
            "is_answered",
            "has_accepted_answer",
            "creation_date",
            "closed_reason",
            "duplicate_of"
        ])
        if new_file:
            w.writeheader()
        w.writerows(rows)

def get_filter_for_questions(api_key):
    params = {
        "include": "question.body;question.closed_details;question.closed_reason",
        "base": "default",
        "unsafe": "false"
    }
    if api_key:
        params["key"] = api_key
    response = requests.post(f"{API}/filters/create", data=params, timeout=30)
    try:
        data = response.json()
    except ValueError:
        raise RuntimeError("Quota exceeded")

    return data["items"][0]["filter"]

def parse_question(question):
    body, code = clean_question_body(question.get("body", ""))
    originals = (question.get("closed_details") or {}).get("original_questions") or []

    return {
        "question_id": question["question_id"],
        "title": BeautifulSoup(question.get("title", ""), "lxml").get_text(),
        "body_text": body,
        "code_text": code,
        "tags": "|".join(question.get("tags", [])),
        "score": question.get("score", 0),
        "view_count": question.get("view_count", 0),
        "answer_count": question.get("answer_count", 0),
        "is_answered": question.get("is_answered", False),
        "has_accepted_answer": "accepted_answer_id" in question,
        "creation_date": datetime.fromtimestamp(question["creation_date"], timezone.utc).isoformat(),
        "closed_reason": question.get("closed_reason", ""),
        "duplicate_of": "|".join(str(original["question_id"]) for original in originals),
    }

## Function to collect data

In [5]:
def collect(start_date, end_date, api_key, output_file):
    try:
        filter_param = get_filter_for_questions(api_key)
    except Exception as e:
        print(f"Could not create API filter ({type(e).__name__}: {e}) - re-run to resume")
        return
    existing_rows = load_seen_question_ids(output_file)
    total = len(existing_rows)
    from_date = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    to_date = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)

    while from_date <= to_date:
        page, has_more, fails = 1, True, 0
        while has_more:
            params = {
                "site": "stackoverflow",
                "sort": "creation",
                "order": "asc",
                "pagesize": 100,
                "page": page,
                "filter": filter_param,
                "fromdate": int(from_date.timestamp()),
                "todate": int((from_date + timedelta(days=1)).timestamp()) - 1
            }
            if api_key:
                params["key"] = api_key
            try:
                response = requests.get(f"{API}/questions", params=params, timeout=30)
                if response.status_code == 429:
                    print(f"Rate limited (HTTP 429) on {from_date.date()} page {page} - stopping. "
                          f"Wait a few minutes, then re-run this cell to resume.")
                    return
                response.raise_for_status()
                try:
                    data = response.json()
                except ValueError:
                    raise RuntimeError(f"non-JSON body (HTTP {response.status_code}): {response.text[:80]!r}")
                if "error_id" in data:
                    raise RuntimeError(f"API error {data.get('error_id')}: {data.get('error_message')}")
                if data.get("backoff"):
                    time.sleep(data["backoff"])  # API asks us to slow down
                rows = []
                for question in data.get("items", []):
                    question_id = str(question["question_id"])
                    if question_id not in existing_rows:
                        rows.append(parse_question(question))
                        existing_rows.add(question_id)
                append_rows(output_file, rows)
                total += len(rows)
                print(f"{from_date.date()} page: {page}: +{len(rows)} rows added (total: {total}, quota left: {data.get('quota_remaining')})")
                if data.get("quota_remaining", 1) == 0:
                    print("Daily quota used - re-run this cell tomorrow to resume")
                    return
                has_more = data.get("has_more", False)
                page += 1
                fails = 0
                time.sleep(1)
            except Exception as e:
                fails += 1
                print(f"{from_date.date()} page: {page} failed ({type(e).__name__}: {e})")
                if fails >= 3:
                    print(f"Skipping rest of {from_date.date()} - set START_DATE to that day to retry it")
                    break
                time.sleep(1)
        from_date += timedelta(days=1)

## Collect the newest data from the last date already collected

In [9]:
if os.path.exists(OUTPUT_CSV_PATH):
    last = pd.to_datetime(pd.read_csv(OUTPUT_CSV_PATH, usecols=["creation_date"])["creation_date"], utc=True).max()
    start = last.date().isoformat() if pd.notna(last) else START_DATE
else:
    start = START_DATE
print(f"Collecting from {start} to {END_DATE}")
collect(start, END_DATE, API_KEY, OUTPUT_CSV_PATH)

2024-07-25 page: 1: +0 rows added (total: 278380, quota left: 7068)
2024-07-25 page: 2: +0 rows added (total: 278380, quota left: 7068)
2024-07-25 page: 3: +0 rows added (total: 278380, quota left: 7066)
2024-07-25 page: 4: +0 rows added (total: 278380, quota left: 7066)
2024-07-25 page: 5: +0 rows added (total: 278380, quota left: 7064)
2024-07-25 page: 6: +0 rows added (total: 278380, quota left: 7064)
2024-07-25 page: 7: +0 rows added (total: 278380, quota left: 7062)
2024-07-25 page: 8: +0 rows added (total: 278380, quota left: 7061)
2024-07-25 page: 9: +0 rows added (total: 278380, quota left: 7061)
2024-07-25 page: 10: +0 rows added (total: 278380, quota left: 7059)
2024-07-25 page: 11: +0 rows added (total: 278380, quota left: 7059)
2024-07-25 page: 12: +0 rows added (total: 278380, quota left: 7057)
2024-07-25 page: 13: +14 rows added (total: 278394, quota left: 7057)
2024-07-26 page: 1: +100 rows added (total: 278494, quota left: 7057)
2024-07-26 page: 2: +100 rows added (tota

/var/folders/jx/vfzctgrd6svfxbn5yt9s35r40000gn/T/ipykernel_84291/842689285.py:62: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  "title": BeautifulSoup(question.get("title", ""), "lxml").get_text(),


2024-07-28 page: 3: +100 rows added (total: 280338, quota left: 7036)
2024-07-28 page: 4: +100 rows added (total: 280438, quota left: 7035)
2024-07-28 page: 5: +100 rows added (total: 280538, quota left: 7035)
2024-07-28 page: 6: +47 rows added (total: 280585, quota left: 7033)
2024-07-29 page: 1: +100 rows added (total: 280685, quota left: 7033)
2024-07-29 page: 2: +100 rows added (total: 280785, quota left: 7031)
2024-07-29 page: 3: +100 rows added (total: 280885, quota left: 7031)
2024-07-29 page: 4: +100 rows added (total: 280985, quota left: 7029)
2024-07-29 page: 5: +100 rows added (total: 281085, quota left: 7029)
2024-07-29 page: 6: +100 rows added (total: 281185, quota left: 7027)
2024-07-29 page: 7: +100 rows added (total: 281285, quota left: 7027)
2024-07-29 page: 8: +100 rows added (total: 281385, quota left: 7025)
2024-07-29 page: 9: +100 rows added (total: 281485, quota left: 7025)
2024-07-29 page: 10: +100 rows added (total: 281585, quota left: 7023)
2024-07-29 page: 11:

## Fetch duplicate targets from the specific tags

In [7]:
def fetch_canonical_targets(api_key, output_file, keep_tags):
    filter_param = get_filter_for_questions(api_key)
    seen = load_seen_question_ids(output_file)
    with open(output_file, newline="", encoding="utf-8") as f:
        targets = set()
        for row in csv.DictReader(f):
            if row["duplicate_of"]:
                targets.update(row["duplicate_of"].split("|"))
    todo = sorted(t for t in targets if t not in seen)
    print(f"Fetching {len(todo)} canonical targets in batches of 100")
    for i in range(0, len(todo), 100):
        batch = todo[i:i + 100]
        params = {"site": "stackoverflow", "filter": filter_param}
        if api_key:
            params["key"] = api_key
        response = requests.get(f"{API}/questions/{';'.join(batch)}", params=params, timeout=30)
        data = response.json()
        rows = []
        for question in data.get("items", []):
            qid = str(question["question_id"])
            if qid not in seen and set(question.get("tags", [])) & keep_tags:
                rows.append(parse_question(question))
                seen.add(qid)
        append_rows(output_file, rows)
        print(f"batch {i // 100}: +{len(rows)} kept (quota left: {data.get('quota_remaining')})")
        if data.get("quota_remaining", 1) == 0:
            print("Daily quota used - re-run to resume.")
            return
        time.sleep(1)
    print("Done.")

In [10]:
fetch_canonical_targets(API_KEY, OUTPUT_CSV_PATH, keep_tags=set(TAGS))

Fetching 15810 canonical targets in batches of 100
batch 0: +3 kept (quota left: 4083)
batch 1: +5 kept (quota left: 4082)
batch 2: +3 kept (quota left: 4082)
batch 3: +9 kept (quota left: 4080)
batch 4: +4 kept (quota left: 4080)
batch 5: +5 kept (quota left: 4078)
batch 6: +6 kept (quota left: 4078)
batch 7: +7 kept (quota left: 4078)
batch 8: +6 kept (quota left: 4076)
batch 9: +7 kept (quota left: 4076)
batch 10: +5 kept (quota left: 4073)
batch 11: +3 kept (quota left: 4073)
batch 12: +5 kept (quota left: 4073)
batch 13: +7 kept (quota left: 4071)
batch 14: +7 kept (quota left: 4069)
batch 15: +1 kept (quota left: 4069)
batch 16: +4 kept (quota left: 4069)
batch 17: +6 kept (quota left: 4066)
batch 18: +7 kept (quota left: 4066)
batch 19: +2 kept (quota left: 4064)
batch 20: +4 kept (quota left: 4064)
batch 21: +6 kept (quota left: 4064)
batch 22: +9 kept (quota left: 4061)
batch 23: +9 kept (quota left: 4061)
batch 24: +9 kept (quota left: 4059)
batch 25: +5 kept (quota left: 405

/var/folders/jx/vfzctgrd6svfxbn5yt9s35r40000gn/T/ipykernel_84291/842689285.py:62: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  "title": BeautifulSoup(question.get("title", ""), "lxml").get_text(),


batch 136: +13 kept (quota left: 3948)
batch 137: +6 kept (quota left: 3946)
batch 138: +12 kept (quota left: 3946)
batch 139: +9 kept (quota left: 3944)
batch 140: +9 kept (quota left: 3944)
batch 141: +10 kept (quota left: 3942)
batch 142: +6 kept (quota left: 3942)
batch 143: +9 kept (quota left: 3942)
batch 144: +12 kept (quota left: 3939)
batch 145: +4 kept (quota left: 3939)
batch 146: +7 kept (quota left: 3937)
batch 147: +8 kept (quota left: 3937)
batch 148: +8 kept (quota left: 3935)
batch 149: +10 kept (quota left: 3935)
batch 150: +11 kept (quota left: 3933)
batch 151: +6 kept (quota left: 3933)
batch 152: +3 kept (quota left: 3931)
batch 153: +10 kept (quota left: 3931)
batch 154: +6 kept (quota left: 3931)
batch 155: +7 kept (quota left: 3928)
batch 156: +5 kept (quota left: 3928)
batch 157: +12 kept (quota left: 3928)
batch 158: +2 kept (quota left: 3925)
Done.


## Verify the data collection

In [6]:
df = pd.read_csv(OUTPUT_CSV_PATH)


df.describe()

,question_id,score,view_count,answer_count
count,1.315189e+06,1.315189e+06,1.315189e+06,1.315189e+06
mean,7.728323e+07,8.133873e-01,9.861632e+02,1.010976e+00
std,2.019821e+06,2.367227e+01,2.785858e+04,9.118194e-01
min,3.061000e+03,-2.600000e+01,5.000000e+00,0.000000e+00
25%,7.611211e+07,0.000000e+00,9.000000e+01,1.000000e+00
50%,7.726232e+07,0.000000e+00,1.850000e+02,1.000000e+00
75%,7.847925e+07,1.000000e+00,5.230000e+02,1.000000e+00
max,8.001862e+07,8.450000e+03,7.568446e+06,7.000000e+01


## References

- https://link.sc/blog/how-to-scrape-stack-overflow
- https://api.stackexchange.com/docs/filters
- https://api.stackexchange.com/docs/questions-by-ids